# ONH Metric Filtering Strategy Selection

This notebook generates report-ready figures to justify the selected metric-specific graph filtering strategy.

The pipeline is fixed:

1. Selected ONH binarisation.
2. Balanced skeletonisation and pruning.
3. Raw node/link extraction using `skel2graph3d`.
4. Compare three metric-filtering strategies only for branchpoint and tortuosity metrics.

The selected setting is **Balanced filtering (15 px minimum link length)** because it reduces inflated graph metrics while retaining enough links for stable analysis.

In [ ]:
from pathlib import Path
import sys

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from scipy.ndimage import convolve, distance_transform_edt
from skimage.morphology import medial_axis, remove_small_objects

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "gui" / "build" / "modules" / "Skel2Graph3D.py").exists():
            if (candidate / "algorithms test images").exists():
                return candidate
    raise FileNotFoundError("Could not find the OCTAVA project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
TEST_DIR = PROJECT_ROOT / "algorithms test images"
GUI_BUILD_DIR = PROJECT_ROOT / "gui" / "build"
MODULE_DIR = GUI_BUILD_DIR / "modules"

for item in (str(GUI_BUILD_DIR), str(MODULE_DIR)):
    if item not in sys.path:
        sys.path.insert(0, item)

from modules.onh_binarise_image import binarise_onh_image
from modules.Skel2Graph3D import skel2graph3d

METADATA_PATH = TEST_DIR / "test images information.xlsx"
OUTPUT_DIR = TEST_DIR / "notebook_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output folder:", OUTPUT_DIR)

## Load ONH Test Images

The four images include clear and blurred examples from two OCTA devices. The annular ROI is fixed at 2-4 mm diameter within a 4.5 mm scan.

In [ ]:
IMAGE_SIZE_PX = 1500
SCAN_SIZE_MM = 4.5
INNER_DIAMETER_MM = 2.0
OUTER_DIAMETER_MM = 4.0

metadata = pd.read_excel(METADATA_PATH)
metadata

In [ ]:
def read_grayscale_image(path: Path) -> np.ndarray:
    return np.asarray(Image.open(path).convert("L"), dtype=np.float64)


def resize_to_scan(image: np.ndarray, image_size_px: int = IMAGE_SIZE_PX) -> np.ndarray:
    interpolation = cv2.INTER_AREA if max(image.shape) > image_size_px else cv2.INTER_CUBIC
    return cv2.resize(image, (image_size_px, image_size_px), interpolation=interpolation).astype(np.float64)


def annulus_mask(image_shape, center_x, center_y):
    h, w = image_shape
    px_per_mm = w / SCAN_SIZE_MM
    inner_radius_px = (INNER_DIAMETER_MM / 2.0) * px_per_mm
    outer_radius_px = (OUTER_DIAMETER_MM / 2.0) * px_per_mm
    yy, xx = np.ogrid[:h, :w]
    distance = np.sqrt((xx - center_x) ** 2 + (yy - center_y) ** 2)
    return (distance >= inner_radius_px) & (distance <= outer_radius_px)


def load_cases(metadata: pd.DataFrame) -> list[dict]:
    cases = []
    for _, row in metadata.iterrows():
        image_path = TEST_DIR / str(row["filename"])
        raw = read_grayscale_image(image_path)
        image = resize_to_scan(raw)
        center_x = float(row["center_x"])
        center_y = float(row["center_y"])
        device = str(row["device"]).strip()
        quality = str(row["quality"]).strip()
        cases.append({
            "name": f"{device} {quality}",
            "image": image,
            "roi_mask": annulus_mask(image.shape, center_x, center_y),
            "center_x": center_x,
            "center_y": center_y,
        })
    return cases


cases = load_cases(metadata)
[(case["name"], case["image"].shape) for case in cases]

## Fixed Image Processing Pipeline

The comparison below does not change binarisation, skeletonisation, or graph extraction. It only changes which graph links are used for branchpoint and tortuosity metrics.

In [ ]:
class NotebookVar:
    def __init__(self, value):
        self.value = value
    def get(self):
        return self.value


class NotebookCanvas:
    status_id = 1
    frangi_bool = NotebookVar(1)
    frangi_min = NotebookVar(1)
    frangi_max = NotebookVar(4)
    med_filt_bool = NotebookVar(0)
    kernel_size = NotebookVar(3)

    def itemconfigure(self, *args, **kwargs):
        pass

    def update_idletasks(self):
        pass


def generate_binary(case: dict) -> np.ndarray:
    canvas = NotebookCanvas()
    canvas.onh_mask_img = case["image"].astype(np.float64)
    canvas.onh_mask_arr = case["roi_mask"].astype(bool)
    binarise_onh_image(canvas, display=False)
    return canvas.binary_image.astype(np.uint8)


def neighbour_count_8(skeleton: np.ndarray) -> np.ndarray:
    kernel = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]], dtype=np.uint8)
    return convolve(skeleton.astype(np.uint8), kernel, mode="constant", cval=0)


def prune_short_spurs(skeleton: np.ndarray, min_branch_length: int = 15, max_iterations: int = 10) -> np.ndarray:
    skel = skeleton.astype(bool).copy()
    for _ in range(max_iterations):
        neighbour_count = neighbour_count_8(skel)
        endpoints = np.argwhere(skel & (neighbour_count == 1))
        if len(endpoints) == 0:
            break

        pixels_to_remove = []
        for endpoint in endpoints:
            path = []
            current = tuple(endpoint)
            previous = None

            for _step in range(min_branch_length + 1):
                path.append(current)
                y, x = current
                neighbours = []
                for dy in (-1, 0, 1):
                    for dx in (-1, 0, 1):
                        if dy == 0 and dx == 0:
                            continue
                        ny, nx = y + dy, x + dx
                        if 0 <= ny < skel.shape[0] and 0 <= nx < skel.shape[1] and skel[ny, nx]:
                            if previous is None or (ny, nx) != previous:
                                neighbours.append((ny, nx))
                if len(neighbours) == 0:
                    break
                if len(neighbours) >= 2:
                    if len(path) <= min_branch_length:
                        pixels_to_remove.extend(path[:-1])
                    break
                previous = current
                current = neighbours[0]

        if not pixels_to_remove:
            break
        for y, x in pixels_to_remove:
            skel[y, x] = False
    return skel


def generate_balanced_skeleton(binary: np.ndarray, roi_mask: np.ndarray) -> np.ndarray:
    roi_mask = roi_mask.astype(bool)
    vessel_mask = (binary > 0) & roi_mask
    image_size = vessel_mask.shape[0]
    pre_min_area = max(20, int(image_size * image_size * 0.00001))
    vessel_mask_clean = remove_small_objects(vessel_mask, min_size=pre_min_area, connectivity=2)
    vessel_mask_clean = vessel_mask_clean & roi_mask
    skeleton_raw, _ = medial_axis(vessel_mask_clean, return_distance=True)
    skeleton_raw = skeleton_raw & roi_mask
    skeleton_component_pruned = remove_small_objects(skeleton_raw, min_size=8, connectivity=2)
    skeleton_component_pruned = skeleton_component_pruned & roi_mask
    skeleton = prune_short_spurs(skeleton_component_pruned, min_branch_length=15, max_iterations=10)
    return skeleton & roi_mask


def build_case_pipeline(case: dict) -> dict:
    binary = generate_binary(case)
    skeleton = generate_balanced_skeleton(binary, case["roi_mask"])
    nodes, links = skel2graph3d(skeleton.astype(bool), 8)
    return {"binary": binary, "skeleton": skeleton, "nodes": nodes, "links": links}


pipeline_results = {case["name"]: build_case_pipeline(case) for case in cases}
print("Generated binary masks, skeletons, and raw graphs.")

## Metric Filtering Strategies

The labels below are designed for the report figure. The selected strategy is **Balanced filtering (15 px)**.

In [ ]:
FILTERING_STRATEGIES = {
    "Unfiltered graph": {
        "min_branch_link_length_px": 0,
        "min_tortuosity_link_length_px": 0,
    },
    "Balanced filtering (15 px)": {
        "min_branch_link_length_px": 15,
        "min_tortuosity_link_length_px": 15,
    },
    "Strict filtering (25 px)": {
        "min_branch_link_length_px": 25,
        "min_tortuosity_link_length_px": 25,
    },
}

FILTERING_STRATEGIES

## Metric Calculation Helpers

Only branchpoint and tortuosity metrics are filtered. The raw graph is preserved.

In [ ]:
def flat_indices_to_rc(indices, shape):
    indices = np.asarray(indices, dtype=int)
    if indices.size == 0:
        return np.array([], dtype=int), np.array([], dtype=int)
    return np.unravel_index(indices, shape)


def valid_link_length_pixels(link, image_shape, roi_mask: np.ndarray) -> float:
    indices = np.asarray(getattr(link, "path", []), dtype=int)
    if indices.size < 2:
        return 0.0
    rows, cols = flat_indices_to_rc(indices, image_shape)
    valid = roi_mask[rows, cols]
    coords = np.column_stack((rows, cols)).astype(float)
    diffs = np.diff(coords, axis=0)
    step_lengths = np.linalg.norm(diffs, axis=1)
    valid_steps = valid[:-1] & valid[1:]
    return float(step_lengths[valid_steps].sum())


def link_tortuosity(link, image_shape, roi_mask: np.ndarray) -> float | None:
    indices = np.asarray(getattr(link, "path", []), dtype=int)
    if indices.size < 3:
        return None
    rows, cols = flat_indices_to_rc(indices, image_shape)
    valid = roi_mask[rows, cols]
    rows = rows[valid]
    cols = cols[valid]
    if rows.size < 3:
        return None
    coords = np.column_stack((rows, cols)).astype(float)
    arc_length = float(np.sum(np.linalg.norm(np.diff(coords, axis=0), axis=1)))
    chord_length = float(np.linalg.norm(coords[-1] - coords[0]))
    if chord_length <= 1e-6 or arc_length < 3:
        return None
    value = arc_length / chord_length
    return float(value) if np.isfinite(value) and 1.0 <= value <= 20.0 else None


def link_table_for_case(links, image_shape, roi_mask: np.ndarray) -> pd.DataFrame:
    rows = []
    for idx, link in enumerate(links):
        rows.append({
            "link_index": idx,
            "start": int(getattr(link, "start", -1)),
            "end": int(getattr(link, "end", -1)),
            "length_px": valid_link_length_pixels(link, image_shape, roi_mask),
            "tortuosity": link_tortuosity(link, image_shape, roi_mask),
        })
    return pd.DataFrame(rows)


def count_filtered_branchpoints(nodes, link_table: pd.DataFrame, image_shape, roi_mask: np.ndarray, min_link_length_px: float) -> int:
    valid_link_indices = set(
        link_table.loc[link_table["length_px"] >= min_link_length_px, "link_index"].astype(int).tolist()
    )
    count = 0
    for node in nodes:
        if getattr(node, "ep", 0) != 0:
            continue
        indices = np.asarray(getattr(node, "nodes_idx", []), dtype=int)
        if indices.size == 0:
            continue
        rows, cols = flat_indices_to_rc(indices, image_shape)
        if not np.any(roi_mask[rows, cols]):
            continue
        connected_valid_links = [idx for idx in getattr(node, "links", []) if int(idx) in valid_link_indices]
        if len(connected_valid_links) >= 2:
            count += 1
    return count


def mean_filtered_tortuosity(link_table: pd.DataFrame, min_link_length_px: float) -> tuple[float, int]:
    valid = link_table[
        (link_table["length_px"] >= min_link_length_px)
        & link_table["tortuosity"].notna()
    ]
    if len(valid) == 0:
        return 0.0, 0
    return float(valid["tortuosity"].mean()), int(len(valid))

## Compare Filtering Strategies

This table is the data source for the report-ready figures.

In [ ]:
def compute_metrics_for_strategy(case: dict, data: dict, strategy_name: str, strategy: dict) -> dict:
    binary = data["binary"]
    skeleton = data["skeleton"]
    nodes = data["nodes"]
    links = data["links"]
    roi_mask = case["roi_mask"].astype(bool)
    image_shape = binary.shape

    pixel_size_mm = SCAN_SIZE_MM / float(image_shape[1])
    pixel_area_mm2 = pixel_size_mm ** 2
    roi_area_mm2 = float(np.count_nonzero(roi_mask) * pixel_area_mm2)

    binary_mask = (binary > 0) & roi_mask
    skeleton_mask = (skeleton > 0) & roi_mask
    vad = float(np.count_nonzero(binary_mask) / max(np.count_nonzero(roi_mask), 1) * 100.0)

    link_table = link_table_for_case(links, image_shape, roi_mask)
    total_length_px = float(link_table["length_px"].sum()) if len(link_table) else float(np.count_nonzero(skeleton_mask))
    total_length_mm = total_length_px * pixel_size_mm
    vld = float(total_length_mm / roi_area_mm2) if roi_area_mm2 > 0 else 0.0

    min_branch = float(strategy["min_branch_link_length_px"])
    min_tort = float(strategy["min_tortuosity_link_length_px"])
    branchpoint_count = count_filtered_branchpoints(nodes, link_table, image_shape, roi_mask, min_branch)
    mean_tort, valid_tort_links = mean_filtered_tortuosity(link_table, min_tort)

    valid_links = int((link_table["length_px"] >= min_branch).sum()) if len(link_table) else 0
    raw_links = int(len(link_table))

    return {
        "case": case["name"],
        "strategy": strategy_name,
        "vessel_area_density_pct": vad,
        "vessel_length_density_mm_per_mm2": vld,
        "branchpoint_count": int(branchpoint_count),
        "branchpoint_density_per_mm2": float(branchpoint_count / roi_area_mm2) if roi_area_mm2 > 0 else 0.0,
        "mean_tortuosity": mean_tort,
        "raw_link_count": raw_links,
        "retained_link_count": valid_links,
        "retained_link_pct": float(valid_links / raw_links * 100.0) if raw_links else 0.0,
        "valid_tortuosity_link_count": valid_tort_links,
        "min_link_length_px": min_branch,
    }


rows = []
for case in cases:
    data = pipeline_results[case["name"]]
    for strategy_name, strategy in FILTERING_STRATEGIES.items():
        rows.append(compute_metrics_for_strategy(case, data, strategy_name, strategy))

metric_comparison = pd.DataFrame(rows)
metric_comparison

## Report Figure 1: Case-Level Metric Comparison

This figure can be used if you want to show the effect of filtering across all four test images.

In [ ]:
REPORT_FIGURE_1 = OUTPUT_DIR / "onh_metric_filtering_case_level_comparison.png"
SELECTED_STRATEGY = "Balanced filtering (15 px)"


def grouped_bar(ax, metric: str, ylabel: str, title: str, ylim=None):
    pivot = metric_comparison.pivot(index="case", columns="strategy", values=metric)
    strategy_order = list(FILTERING_STRATEGIES.keys())
    pivot = pivot[strategy_order]
    colors = ["#9aa0a6", "#2f80ed", "#c75b5b"]
    pivot.plot(kind="bar", ax=ax, width=0.78, color=colors)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=0)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.legend_.remove()


fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
grouped_bar(axes[0], "branchpoint_density_per_mm2", "Branchpoints / mm²", "Branchpoint density")
grouped_bar(axes[1], "mean_tortuosity", "Mean tortuosity", "Tortuosity")
grouped_bar(axes[2], "retained_link_pct", "Retained links (%)", "Links retained for graph-sensitive metrics", ylim=(0, 110))

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
fig.suptitle("ONH metric filtering strategy comparison", y=1.04)
fig.tight_layout(rect=(0, 0.12, 1, 1))
fig.savefig(REPORT_FIGURE_1, bbox_inches="tight")
plt.show()
print("Saved:", REPORT_FIGURE_1)

## Report Figure 2: Decision Summary

This is the most compact figure for the report. It shows why the 15 px strategy was selected: it reduces branchpoint inflation while keeping enough graph links. The 25 px strategy is cleaner but removes too much graph information.

In [ ]:
REPORT_FIGURE_2 = OUTPUT_DIR / "onh_metric_filtering_selection_summary.png"

baseline = metric_comparison[metric_comparison["strategy"] == "Unfiltered graph"].set_index("case")
summary_rows = []
for strategy_name in ["Balanced filtering (15 px)", "Strict filtering (25 px)"]:
    current = metric_comparison[metric_comparison["strategy"] == strategy_name].set_index("case")
    branchpoint_reduction_pct = (
        (baseline["branchpoint_count"] - current["branchpoint_count"])
        / baseline["branchpoint_count"].replace(0, np.nan)
        * 100.0
    )
    tortuosity_change_pct = (
        (current["mean_tortuosity"] - baseline["mean_tortuosity"])
        / baseline["mean_tortuosity"].replace(0, np.nan)
        * 100.0
    )
    summary_rows.append({
        "strategy": strategy_name,
        "Branchpoint reduction (%)": float(branchpoint_reduction_pct.mean()),
        "Retained links (%)": float(current["retained_link_pct"].mean()),
        "Tortuosity change (%)": float(tortuosity_change_pct.mean()),
    })

selection_summary = pd.DataFrame(summary_rows)
selection_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.8))
metrics_to_plot = ["Branchpoint reduction (%)", "Retained links (%)", "Tortuosity change (%)"]
x = np.arange(len(metrics_to_plot))
width = 0.34
colors = ["#2f80ed", "#c75b5b"]

for i, (_, row) in enumerate(selection_summary.iterrows()):
    values = [row[m] for m in metrics_to_plot]
    bars = ax.bar(x + (i - 0.5) * width, values, width, label=row["strategy"], color=colors[i])
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 1.5,
            f"{height:.1f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylabel("Mean change across four test images (%)")
ax.set_title("Selection of metric-specific graph filtering threshold")
ax.set_ylim(0, max(105, selection_summary[metrics_to_plot].to_numpy().max() + 12))
ax.legend(frameon=False, loc="upper right")
ax.text(
    0.02,
    0.94,
    "Selected: Balanced filtering (15 px)",
    transform=ax.transAxes,
    fontsize=11,
    color="#2f80ed",
    fontweight="bold",
    va="top",
)
fig.tight_layout()
fig.savefig(REPORT_FIGURE_2, bbox_inches="tight")
plt.show()
print("Saved:", REPORT_FIGURE_2)

## Short Report Table

This optional table gives the numeric support for the selected 15 px filtering threshold.

In [ ]:
report_table = metric_comparison[[
    "case",
    "strategy",
    "branchpoint_count",
    "branchpoint_density_per_mm2",
    "mean_tortuosity",
    "retained_link_count",
    "retained_link_pct",
    "valid_tortuosity_link_count",
]].copy()
report_table

Suggested caption for **Figure: Selection of metric-specific graph filtering threshold**:

**Figure X. Selection of metric-specific graph filtering threshold.** Balanced filtering with a 15 px minimum link length reduced branchpoint inflation while retaining more than half of the extracted links for graph-sensitive metrics. Strict 25 px filtering removed more short links, but retained substantially fewer links and was therefore considered too aggressive for the ONH capillary network.